# Gene2Image — 关键步骤可视化

本 notebook 可视化通路编码器的关键中间步骤，便于理解与调试：
1. **通路掩码 A** 的结构（稀疏度、每通路基因数、真实 vs 随机 vs 全1）
2. **通路 token 嵌入**（模块 A 输出）
3. **CLS→通路注意力**（RQ4 可解释性信号）
4. **生成样本**（真实 vs 生成 H&E）

运行环境：`zw@Gene2Image`。从 `code/` 目录启动 jupyter。

In [ ]:
import os, sys, json
import numpy as np
import torch
import matplotlib.pyplot as plt

# 确保从 code/ 根目录导入
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## 1 通路掩码结构

查看 C1 的 Hallmark 掩码：稀疏度、每通路命中基因数，以及 real / rand / none 三变体的差异。

In [ ]:
DS = 'c1'
real = np.load(f'data/pathway_masks/{DS}_hallmark_real.npz', allow_pickle=True)
rand = np.load(f'data/pathway_masks/{DS}_hallmark_rand.npz', allow_pickle=True)
none = np.load(f'data/pathway_masks/{DS}_hallmark_none.npz', allow_pickle=True)
A = real['A']; pnames = list(real['pathway_names']); gnames = list(real['gene_names'])
print(f'P={A.shape[0]} pathways, G={A.shape[1]} genes, edges={int(A.sum())}, density={A.mean():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, M) in zip(axes, [('real', real['A']), ('rand', rand['A']), ('none', none['A'])]):
    ax.imshow(M, aspect='auto', cmap='Greys', interpolation='nearest')
    ax.set_title(f'{name} mask  (sum={int(M.sum())})')
    ax.set_xlabel('gene'); ax.set_ylabel('pathway')
plt.tight_layout(); plt.show()

In [ ]:
# 每通路命中基因数分布；real 与 rand 应一致（同密度）
per_real = real['A'].sum(1); per_rand = rand['A'].sum(1)
order = np.argsort(per_real)[::-1]
plt.figure(figsize=(12, 4))
plt.bar(range(len(per_real)), per_real[order], width=0.8, label='real')
plt.xticks(range(len(pnames)), [pnames[i] for i in order], rotation=90, fontsize=6)
plt.ylabel('genes per pathway'); plt.title(f'{DS} Hallmark: genes per pathway'); plt.legend()
plt.tight_layout(); plt.show()
print('real==rand per-pathway counts:', bool((np.sort(per_real)==np.sort(per_rand)).all()))

## 2 通路 token 嵌入（模块 A）

用真实 C1 表达喂入 `PathwayMaskEmbedding`，可视化每个细胞的 [P, d_token] 通路 token。

In [ ]:
from src.pathway_encoder import PathwayMaskEmbedding
import anndata as ad

adata = ad.read_h5ad('data/processed_data/Xenium_V1_hSkin_Melanoma_Base_FFPE/adata.h5ad', backed='r')
# 取前若干细胞的表达（确保基因顺序 == 掩码列顺序）
assert list(adata.var_names) == gnames, 'gene order mismatch between adata and mask!'
X = adata.X[:8]
X = X.toarray() if hasattr(X, 'toarray') else np.asarray(X)
x = torch.tensor(X, dtype=torch.float32)

embed = PathwayMaskEmbedding(torch.tensor(A, dtype=torch.float32), d_token=48, learnable=True)
T = embed(x)  # [8, P, 48]
print('pathway tokens:', tuple(T.shape))

plt.figure(figsize=(10, 4))
plt.imshow(T[0].detach().numpy(), aspect='auto', cmap='coolwarm')
plt.colorbar(); plt.xlabel('d_token'); plt.ylabel('pathway'); plt.title('cell 0: pathway tokens [P, 48]')
plt.tight_layout(); plt.show()

## 3 CLS→通路注意力（RQ4）

加载训练好的 Gene2Image checkpoint，提取每个细胞的通路注意力谱。
> 修改 `CKPT` 为你的真实训练 checkpoint。未训练模型注意力近似均匀（熵≈log P）。

In [ ]:
from src.single_model import RNAtoHnEModel

CKPT = 'results/gene2image_c1_seed42/checkpoints/best_checkpoint.pt'  # <-- 改成你的 checkpoint
if not os.path.exists(CKPT):
    print('checkpoint 不存在，跳过；请先训练或修改 CKPT 路径'); model = None
else:
    ck = torch.load(CKPT, map_location='cpu', weights_only=False); cfg = ck['config']
    mask = cfg['pathway_mask_array']; mask = mask if torch.is_tensor(mask) else torch.tensor(np.asarray(mask))
    model = RNAtoHnEModel(rna_dim=mask.shape[1], img_channels=4, img_size=256, model_channels=128,
        num_res_blocks=2, attention_resolutions=(16,), channel_mult=(1,2,2,2), num_heads=2, num_head_channels=16,
        encoder_type='pathway', pathway_mask=mask.float(), d_token=cfg['d_token'],
        pt_layers=cfg['pt_layers'], pt_heads=cfg['pt_heads'],
        learnable_pathway=cfg['learnable_pathway'], use_pathway_transformer=cfg['use_pathway_transformer'])
    state = {k.replace('module.', ''): v for k, v in ck.get('model_state_dict', ck.get('model')).items()}
    model.load_state_dict(state); model.to(DEVICE).eval()
    print('loaded', CKPT)

In [ ]:
if model is not None:
    attn = model.rna_encoder.get_pathway_attention(x.to(DEVICE)).cpu().numpy()  # [8, P]
    ent = -(attn * np.log(attn + 1e-12)).sum(1)
    print('attention entropy (lower=more focused):', ent.round(3), ' | uniform=', round(float(np.log(A.shape[0])), 3))
    plt.figure(figsize=(12, 4))
    plt.imshow(attn, aspect='auto', cmap='viridis')
    plt.colorbar(); plt.xlabel('pathway'); plt.ylabel('cell'); plt.title('CLS->pathway attention [cells, P]')
    plt.tight_layout(); plt.show()
    # 平均主导通路
    mean_a = attn.mean(0); top = mean_a.argsort()[::-1][:10]
    print('top-10 dominant pathways:'); [print(f'  {pnames[i]}: {mean_a[i]:.3f}') for i in top]

## 4 生成样本（真实 vs 生成）

训练运行会在 `results/<run>/generation_results.png` 与 `generated_images/` 自动保存对比图，直接展示。

In [ ]:
import matplotlib.image as mpimg
gen_png = 'results/gene2image_c1_seed42/generation_results.png'  # <-- 改成你的 run
if os.path.exists(gen_png):
    plt.figure(figsize=(16, 8)); plt.imshow(mpimg.imread(gen_png)); plt.axis('off'); plt.show()
else:
    print('generation_results.png 不存在；训练完成后会自动生成于 results/<run>/')

In [ ]:
# 训练损失曲线
import pandas as pd
csv = 'results/gene2image_c1_seed42/training_losses.csv'
if os.path.exists(csv):
    df = pd.read_csv(csv)
    plt.figure(figsize=(8, 4))
    plt.plot(df['epoch'], df['train_loss'], label='train')
    plt.plot(df['epoch'], df['val_loss'], label='val')
    plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title('Gene2Image C1 training')
    plt.tight_layout(); plt.show()
else:
    print('training_losses.csv 不存在')